&nbsp;

# Metadata Concat Process

- RNA library の名寄せもここで行う

In [1]:
import datetime

import polars as pl
import pathlib
from IPython.display import display

pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_cols(200)
pl.Config.set_tbl_rows(4)

polars.config.Config

In [2]:
VERSION = datetime.datetime.now().strftime("%y%m")
execute_date = datetime.datetime.now().strftime("%y%m%d")
print(f"Execute Date: {execute_date}")

# create directory
pathlib.Path(f"../../Data/metadata/metadata_{VERSION}").mkdir(parents=True, exist_ok=True)

Execute Date: 261001


&nbsp;

&nbsp;

&nbsp;

## RNA library

監査レポート「3. rna_library の表記統一案」に基づく表記辞書を作る。

- 原記載 44 種類（空欄と unknown を含む）を、表記だけの統一で 30 種類にする
- 大文字・小文字、商標記号、kit / Kit、TrueSeq / TruSeq を揃える
- Stranded / Directional、Ultra / Ultra II、mRNA / Total RNA、LT、FWD、rRNA 除去、poly(A) 選択の有無は別ラベルのまま残す
- 抽出・精製キットと委託先は、作製キットと種類を分ける
- 文献照合による訂正は、この表記辞書より監査一覧を優先する

In [3]:
# Notation-only map from audit report section 3.
# Literature corrections in section 2 override this map and are not applied here.
# The em dash in the report is a missing original value, stored as null.
# U+2032 PRIME is kept in the QuantSeq label. Trailing space on the Zymo kit name is part of the recorded string.
rna_library_notation_rows = [
    # TruSeq Stranded 
    {
        "original_rna_library": "TruSeq Stranded Total RNA Library Prep Kit",
        "normalized_label": "TruSeq Stranded Total RNA Library Prep Kit",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-total-rna-workflow/truseq-stranded-total-rna-workflow-reference-1000000040499-00.pdf"
    },
    {
        "original_rna_library": "TruSeq stranded total RNA protocol",
        "normalized_label": "TruSeq Stranded Total RNA Library Prep Kit",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-total-rna-workflow/truseq-stranded-total-rna-workflow-reference-1000000040499-00.pdf"
    },

    {
        "original_rna_library": "Illumina TruSeq Stranded mRNA Sample Preparation Guide",
        "normalized_label": "TruSeq Stranded mRNA Library Prep Kit",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-mrna-workflow/truseq-stranded-mrna-workflow-reference-1000000040498-00.pdf"
    },
    {
        "original_rna_library": "TruSeq Stranded mRNA Library Prep Kit",
        "normalized_label": "TruSeq Stranded mRNA Library Prep Kit",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-mrna-workflow/truseq-stranded-mrna-workflow-reference-1000000040498-00.pdf",
    },
    {
        "original_rna_library": "TruSeq Stranded mRNA Library Kit",
        "normalized_label": "TruSeq Stranded mRNA Library Prep Kit",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-mrna-workflow/truseq-stranded-mrna-workflow-reference-1000000040498-00.pdf"
    },
    {
        "original_rna_library": "TruSeq Stranded mRNA Library",
        "normalized_label": "TruSeq Stranded mRNA Library Prep Kit",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-mrna-workflow/truseq-stranded-mrna-workflow-reference-1000000040498-00.pdf",
    },
    {
        "original_rna_library": "TruSeq Stranded mRNA LT Sample Prep Kit",
        "normalized_label": "TruSeq Stranded mRNA LT Sample Prep Kit",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://ostr.ccr.cancer.gov/wp-content/uploads/2020/05/truseq-stranded-mrna-sample-prep-guide-15031047-e.pdf",
    },
    {
        "original_rna_library": "TruSeq stranded mRNA library preparation kit",
        "normalized_label": "TruSeq Stranded mRNA Library Prep Kit",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-mrna-workflow/truseq-stranded-mrna-workflow-reference-1000000040498-00.pdf",
    },
    {
        "original_rna_library": "TruSeq Stranded Total RNA Library Prep Kit (without rRNA depletion)",
        "normalized_label": "TruSeq Stranded Total RNA Library Prep Kit (without rRNA depletion)",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-total-rna-workflow/truseq-stranded-total-rna-workflow-reference-1000000040499-00.pdf",
    },
    {
        "original_rna_library": "TruSeq Stranded mRNA Library Prep Kit (without poly(A) selection)",
        "normalized_label": "TruSeq Stranded mRNA Library Prep Kit (without poly(A) selection)",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-mrna-workflow/truseq-stranded-mrna-workflow-reference-1000000040498-00.pdf",
    },
    {
        "original_rna_library": "TruSeq Stranded mRNA assay",
        "normalized_label": "TruSeq Stranded mRNA Library Prep Kit",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-mrna-workflow/truseq-stranded-mrna-workflow-reference-1000000040498-00.pdf",
    },

    # NEBNext Ultra II Directional RNA Library Prep Kit
    {
        "original_rna_library": "NEBNext Ultra II Directional RNA Library Prep Kit",
        "normalized_label": "NEBNext Ultra II Directional RNA Library Prep Kit for Illumina",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://www.neb.com/en-us/products/e7760-nebnext-ultra-ii-directional-rna-library-prep-kit-for-illumina?srsltid=AU7gw4Xlae0blQQ89Otcs2wOzZiRMVMtmTjsIIuOrzEuHSJgMt5ryFNt"
    },
    {
        "original_rna_library": "E7765L NEBNext Ultra II Directional RNA Library Prep with Beads",
        "normalized_label": "NEBNext Ultra II Directional RNA Library Prep with Sample Purification Beads (E7765L)",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://www.neb.com/en-us/-/media/nebus/files/manuals/manuale7760_e7765.pdf?rev=a5a99045a0bf4ee8afaf80fd2e3653dc&hash=9914BA403C3FD9B5D3D0C9670DF8C765"
    },
    {
        "original_rna_library": "NEBNext Ultra II Directional RNA library preparation kit",
        "normalized_label": "NEBNext Ultra II Directional RNA Library Prep Kit for Illumina",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://www.neb.com/en/products/e7760-nebnext-ultra-ii-directional-rna-library-prep-kit-for-illumina",
    },
    {
        "original_rna_library": "NEBNext® UltraTM Directional RNA Library Prep Kit",
        "normalized_label": "NEBNext Ultra Directional RNA Library Prep Kit for Illumina",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://www.neb.com/en/products/e7420-nebnext-ultra-directional-rna-library-prep-kit-for-illumina",
    },
    {
        "original_rna_library": "NEBNext Ultra II Directional RNA Library Prep Kit for Illumina",
        "normalized_label": "NEBNext Ultra II Directional RNA Library Prep Kit for Illumina",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://www.neb.com/en/products/e7760-nebnext-ultra-ii-directional-rna-library-prep-kit-for-illumina",
    },

    # KAPA Stranded mRNA-Seq Kit
    {
        "original_rna_library": "KAPA mRNA HyperPrep Kit",
        "normalized_label": "KAPA mRNA HyperPrep Kit",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://elabdoc-prod.roche.com/eLD/web/global/en/products/SEQ-KAPA-0164?searchTerm=8098115702&catalog=ProductCatalog&orderBy=Relevance&fromResults=true",
    },
    {
        "original_rna_library": "KAPA Stranded mRNA-Seq Kit",
        "normalized_label": "KAPA Stranded mRNA-Seq Kit",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://rochesequencingstore.com/wp-content/uploads/2022/07/KAPA-Stranded-mRNA-Seq-Kit-Technical-Data-Sheet.pdf",
    },
    {
        "original_rna_library": "Hyper Prep RNA sequencing kit",
        "normalized_label": "KAPA RNA HyperPrep Kit",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://rochesequencingstore.com/catalog/kapa-rna-hyperprep-kit/",
    },

    # Other
    {
        "original_rna_library": "SMARTer Stranded Total RNA-Seq Kit v2",
        "normalized_label": "SMARTer Stranded Total RNA-Seq Kit v2",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://www.takarabio.com/products/next-generation-sequencing/rna-seq/legacy-rna-seq-kits/pico-input-strand-specific-total-rna-seq-for-mammalian-samples-v2",
    },
    {
        "original_rna_library": "SMARTer® Stranded Total RNA-Seq Kit v2—Pico Input Mammalian (634413, Takara – Clontech, Japan)",
        "normalized_label": "SMARTer Stranded Total RNA-Seq Kit v2",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://www.takarabio.com/products/next-generation-sequencing/rna-seq/legacy-rna-seq-kits/pico-input-strand-specific-total-rna-seq-for-mammalian-samples-v2",
    },
    {
        "original_rna_library": "Lexogen QuantSeq FWD Kit",
        "normalized_label": "QuantSeq 3\u2032 mRNA-Seq Library Prep Kit FWD for Illumina",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://www.lexogen.com/quantseq-3mRNA%20sequencing/",
    },
    {
        "original_rna_library": "Ovation Universal RNA-Seq System",
        "normalized_label": "Ovation Universal RNA-Seq System",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://doi.org/10.1038/s41467-019-11536-8"
    },
    {
        "original_rna_library": "SureSelect Strand-Specific RNA Library Preparation Kits",
        "normalized_label": "SureSelect Strand-Specific RNA Library Preparation Kit",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://www.agilent.com/cs/library/usermanuals/public/G9691-90010.pdf",
    },

    # MSR-seq (custom mRNA library preparation) (strandness is inferred from the reference paper)
    {
        "original_rna_library": "MSR-seq (custom mRNA library preparation)",
        "normalized_label": "MSR-seq (custom mRNA library preparation)",
        "item_type": "library_preparation",
        "strandness": "stranded",
        "strandness_evidence": "https://doi.org/10.1038/s41467-022-30261-3",
    },


    # unknown / unstranded / conflict / mixed_components
    {
        "original_rna_library": "NEBNext® UltraTM RNA Library Prep Kit for Illumina® (NEB, USA)",
        "normalized_label": "NEBNext Ultra RNA Library Prep Kit for Illumina",
        "item_type": "library_preparation",
        "strandness": "unstranded",
        "strandness_evidence": "https://www.neb.com/en-gb/products/e7530-nebnext-ultra-rna-library-prep-kit-for-illumina",
    },
    {
        "original_rna_library": "NEBNext Ultra II RNA Library Prep Kit for Illumina",
        "normalized_label": "NEBNext Ultra II RNA Library Prep Kit for Illumina",
        "item_type": "library_preparation",
        "strandness": "unstranded",
        "strandness_evidence": "https://www.neb.com/en/products/e7770-nebnext-ultra-ii-rna-library-prep-kit-for-illumina",
    },

    {
        "original_rna_library": "VAHTS Universal V6 RNA-seq Library Prep Kit",
        "normalized_label": "VAHTS Universal V6 RNA-seq Library Prep Kit",
        "item_type": "library_preparation",
        "strandness": "unknown",
        "strandness_evidence": "https://www.lifesct.com/download/Manuals/NR604-VAHTS-Universal-V6-RNA-seq-Library-Prep-Kit.pdf",
    },

    {
        "original_rna_library": "TruSeq RNA Sample Preparation Kit",
        "normalized_label": "TruSeq RNA Sample Preparation Kit (version unspecified)",
        "item_type": "library_preparation",
        "strandness": "unknown",
        "strandness_evidence": "",
    },
    {
        "original_rna_library": "VAHTS mRNA-seq v2 Library Prep Kit for Illumina",
        "normalized_label": "VAHTS mRNA-seq v2 Library Prep Kit for Illumina",
        "item_type": "library_preparation",
        "strandness": "unknown",
        "strandness_evidence": "",
    },

    {
        "original_rna_library": "TruSeq mRNA Library Prep Kit",
        "normalized_label": "TruSeq mRNA Library Prep Kit",
        "item_type": "library_preparation",
        "strandness": "unknown",
        "strandness_evidence": "",
    },
    {
        "original_rna_library": "TruSeq stranded mRNA library prep kit (Illumina) (from GEO) / Illumina TruSeq RNA Sample Prep v2 Kit (from article)",
        "normalized_label": "conflict",
        "item_type": "library_preparation",
        "strandness": "conflict",
        "strandness_evidence": "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM6914924; https://doi.org/10.1038/s41467-023-43262-7",
    },
    {
        "original_rna_library": "Illumina protocol",
        "normalized_label": "Illumina protocol (specific kit unspecified)",
        "item_type": "library_preparation",
        "strandness": "unknown",
        "strandness_evidence": "",
    },
    {
        "original_rna_library": (
            "VAHTS universal V8 RNA-seq library prep kit for Illumina "
            "(Vazyme, cat. no. NR605) and KAPA HyperPrep kit (Roche, cat. no. KR0961)"
        ),
        "normalized_label": (
            "VAHTS Universal V8 RNA-seq Library Prep Kit for Illumina "
            "(Vazyme, cat. no. NR605) & KAPA HyperPrep kit (Roche, cat. no. KR0961)"
        ),
        "item_type": "mixed_components",
        "strandness": "unknown",
        "strandness_evidence": "",
    },
    {
        "original_rna_library": "Truseq RNA prep kit",
        "normalized_label": "TruSeq RNA (product/version unspecified)",
        "item_type": "library_preparation",
        "strandness": "unknown",
        "strandness_evidence": "",
    },
    {
        "original_rna_library": "Illumina kit (TruSeq RNA)",
        "normalized_label": "TruSeq RNA (product/version unspecified)",
        "item_type": "library_preparation",
        "strandness": "unknown",
        "strandness_evidence": "",
    },
    {
        "original_rna_library": "mRNA Seq Sample Preparation Kit",
        "normalized_label": "mRNA Seq Sample Preparation Kit",
        "item_type": "library_preparation",
        "strandness": "unknown",
        "strandness_evidence": "",
    },
    {
        "original_rna_library": "Illumina TruSeq RNA sample preparation guide (specific kit/version unspecified)",
        "normalized_label": "TruSeq RNA (product/version unspecified)",
        "item_type": "library_preparation",
        "strandness": "unknown",
        "strandness_evidence": "",
    },
    {
        "original_rna_library": "unknown",
        "normalized_label": "unknown",
        "item_type": "missing",
        "strandness": "unknown",
        "strandness_evidence": "",
    },
    {
        "original_rna_library": None,
        "normalized_label": "unknown",
        "item_type": "missing",
        "strandness": "unknown",
        "strandness_evidence": "",
    }
]

rna_library_notation = pl.DataFrame(rna_library_notation_rows)

# None is the key for a blank original value.
rna_library_notation_dict = dict(
    zip(
        rna_library_notation["original_rna_library"].to_list(),
        rna_library_notation["normalized_label"].to_list(),
        strict=True,
    )
)

# if rna_library_notation.height != 43:
#     raise ValueError(f"expected 44 original labels, got {rna_library_notation.height}")
# if rna_library_notation["original_rna_library"].n_unique() != 43:
#     raise ValueError("original labels are not unique")
# if rna_library_notation["normalized_label"].n_unique() != 30:
#     raise ValueError(
#         f"expected 30 normalized labels, got {rna_library_notation['normalized_label'].n_unique()}"
#     )
# if len(rna_library_notation_dict) != 43:
#     raise ValueError("notation dict dropped a key")

notation_summary = rna_library_notation.group_by("item_type").agg(
    pl.len().alias("n_original_labels"),
    pl.col("normalized_label").n_unique().alias("n_normalized_labels"),
).sort("item_type")


display(rna_library_notation)
display(notation_summary)

original_rna_library,normalized_label,item_type,strandness,strandness_evidence
str,str,str,str,str
"""TruSeq Stranded Total RNA Library Prep Kit""","""TruSeq Stranded Total RNA Library Prep Kit""","""library_preparation""","""stranded""","""https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-total-rna-workflow/truseq-stranded-total-rna-workflow-refer…"
"""TruSeq stranded total RNA protocol""","""TruSeq Stranded Total RNA Library Prep Kit""","""library_preparation""","""stranded""","""https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-total-rna-workflow/truseq-stranded-total-rna-workflow-refer…"
…,…,…,…,…
"""unknown""","""unknown""","""missing""","""unknown""",""""""
null,"""unknown""","""missing""","""unknown""",""""""


item_type,n_original_labels,n_normalized_labels
str,u32,u32
"""library_preparation""",37,26
"""missing""",2,1
"""mixed_components""",1,1


&nbsp;

&nbsp;

## 1. Concat Process (2026-09-30)

- bulk RNA-seqデータのメタデータをconcatする
- 一旦全てのカラムを含めて `concat`

In [4]:
true_control = pl.read_csv(
    f"../../Data/metadata/metadata_{VERSION}/rna_seq_true_control_metadata_{execute_date}.tsv",
    separator="\t",
    ignore_errors=True
)

true_no_control = pl.read_csv(
    f"../../Data/metadata/metadata_{VERSION}/rna_seq_true_no_control_metadata_{execute_date}.tsv",
    separator="\t",
    ignore_errors=True
)

false_control = pl.read_csv(
    f"../../Data/metadata/metadata_{VERSION}/rna_seq_false_control_metadata_{execute_date}.tsv",
    separator="\t",
    ignore_errors=True
)

false_no_control = pl.read_csv(
    f"../../Data/metadata/metadata_{VERSION}/rna_seq_false_no_control_metadata_{execute_date}.tsv",
    separator="\t",
    ignore_errors=True
)

bioprolect_biostudies = pl.read_csv(
    f"../../Data/metadata/metadata_{VERSION}/rna_seq_bioproject_biostudies_metadata_{execute_date}.tsv",
    separator="\t",
    ignore_errors=True
)

manual_curation = pl.read_csv(
    f"../../Data/metadata/metadata_{VERSION}/rna_seq_manual_curation_metadata_{execute_date}.tsv",
    separator="\t",
    ignore_errors=True
)

In [5]:
rna_seq_all_metadata = pl.concat(
    [
        true_control,
        true_no_control,
        false_control,
        false_no_control,
        bioprolect_biostudies,
        manual_curation
    ],
    how="diagonal_relaxed"
)

rna_seq_all_metadata = rna_seq_all_metadata[[s.name for s in rna_seq_all_metadata if not (s.null_count() == rna_seq_all_metadata.height)]]

rna_seq_all_metadata.write_csv(
    f"../../Data/metadata/metadata_{VERSION}/rna_seq_all_metadata_{execute_date}_no_filter.tsv",
    separator="\t"
)

display(rna_seq_all_metadata)

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,genotype,treatment,mapping,expression_estimation,deg_analysis,assembly,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias,sample_data_processing,sample_contact_email,tissue,biological_replicates,tissue_type,protocol,organism,read_type,data_source,srr,srx,sample_contact_department,passage,phenotype,replicate,genome_build,center_name,sample_contact_laboratory,sample_supplementary_file_2,sample_supplementary_file_3,disease,stress,sample_contact_phone,tumor,sex,age,sample_treatment_protocol_ch1,sample_group,cell_model,sample_extract_protocol_ch1,time,library_strategy_right,sample_growth_protocol_ch1,sirna,condition,tissue_cells,antibody,batch,column,library_name_right,sample_characteristics_ch1,centrifuge,strain,fraction,enzyme_chemical_treatment,rt_condition,strain_right,treatment_conditions,molecule,isolate,isolate_right,dosage,experiment,timepoint,vector,sirna_sgrna,sra_file_role,fastq_file_role,tag,scientific_name,tax_id,accession,collection_date_end,collection_date_start,submitted_galaxy,submitted_ftp,submitted_aspera,submitted_md5,checklist,submitted_bytes,nominal_sdev,broker_name,datahub,nominal_length,submitted_read_type,submitted_format,bam_ftp,bam_bytes,dev_stage,bam_aspera,bam_galaxy,bam_file_role,submitted_file_role,recovery treatment temperature,recovery treatment time,sample_series_id,biosample,sra,fastq_galaxy,fastq_ftp,fastq_aspera,tax_lineage,Read length for mate1(bp),Read length for mate 2(bp),Individual Accession,Biomaterial Provider
str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str
"""SRR27681984""","""SRR27681986""","""wi-38-htert_40°c_rna-seq_rep1 vs wi-38-htert_37°c_rna-seq_rep1""","""WI-38-hTERT, 40°C, RNA-seq, rep1 vs WI-38-hTERT, 37°C, RNA-seq, rep1""","""GSM8029688 vs GSM8029686""","""Public on Oct 15 2024""","""Jan 22 2024""","""Oct 15 2024""","""SRA""",1,"""WI-38-hTERT""","""Homo sapiens""",9606,"""total RNA""","""40C_RNA-seq_rep1, 2019_005A_40C_RNA-seq_rep1 vs 37C_RNA-seq_rep1, 2019_005A_37C_RNA-seq_rep1""","""GPL11154""","""Yoshikazu,,Na

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. Strandness

- strandness 情報を結合する

In [6]:
rna_library_notation4join = rna_library_notation.rename(
    {
        "original_rna_library": "rna_library"
    }
)

display(rna_library_notation4join)

rna_library,normalized_label,item_type,strandness,strandness_evidence
str,str,str,str,str
"""TruSeq Stranded Total RNA Library Prep Kit""","""TruSeq Stranded Total RNA Library Prep Kit""","""library_preparation""","""stranded""","""https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-total-rna-workflow/truseq-stranded-total-rna-workflow-refer…"
"""TruSeq stranded total RNA protocol""","""TruSeq Stranded Total RNA Library Prep Kit""","""library_preparation""","""stranded""","""https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-total-rna-workflow/truseq-stranded-total-rna-workflow-refer…"
…,…,…,…,…
"""unknown""","""unknown""","""missing""","""unknown""",""""""
null,"""unknown""","""missing""","""unknown""",""""""


In [7]:
rna_seq_all_metadata_library_notation = rna_seq_all_metadata.join(
    rna_library_notation4join,
    on="rna_library",
    how="left"
)

display(rna_seq_all_metadata_library_notation)

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,genotype,treatment,mapping,expression_estimation,deg_analysis,assembly,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias,sample_data_processing,sample_contact_email,tissue,biological_replicates,tissue_type,protocol,organism,read_type,data_source,srr,srx,sample_contact_department,passage,phenotype,replicate,genome_build,center_name,sample_contact_laboratory,sample_supplementary_file_2,sample_supplementary_file_3,disease,stress,sample_contact_phone,tumor,sex,age,sample_treatment_protocol_ch1,sample_group,cell_model,sample_extract_protocol_ch1,time,library_strategy_right,sample_growth_protocol_ch1,sirna,condition,tissue_cells,antibody,batch,column,library_name_right,sample_characteristics_ch1,centrifuge,strain,fraction,enzyme_chemical_treatment,rt_condition,strain_right,treatment_conditions,molecule,isolate,isolate_right,dosage,experiment,timepoint,vector,sirna_sgrna,sra_file_role,fastq_file_role,tag,scientific_name,tax_id,accession,collection_date_end,collection_date_start,submitted_galaxy,submitted_ftp,submitted_aspera,submitted_md5,checklist,submitted_bytes,nominal_sdev,broker_name,datahub,nominal_length,submitted_read_type,submitted_format,bam_ftp,bam_bytes,dev_stage,bam_aspera,bam_galaxy,bam_file_role,submitted_file_role,recovery treatment temperature,recovery treatment time,sample_series_id,biosample,sra,fastq_galaxy,fastq_ftp,fastq_aspera,tax_lineage,Read length for mate1(bp),Read length for mate 2(bp),Individual Accession,Biomaterial Provider,normalized_label,item_type,strandness,strandness_evidence
str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,str,str,str,str
"""SRR27681984""","""SRR27681986""","""wi-38-htert_40°c_rna-seq_rep1 vs wi-38-htert_37°c_rna-seq_rep1""","""WI-38-hTERT, 40°C, RNA-seq, rep1 vs WI-38-hTERT, 37°C, RNA-seq, rep1""","""GSM8029688 vs GSM8029686""","""Public on Oct 15 2024""","""Jan 22 2024""","""Oct 15 2024""","""SRA""",1,"""WI-38-hTERT""","""Homo sapiens""",9606,"""total RNA""","""40C_RNA-seq_rep1, 2019_005A_40C_RNA-seq_rep1 vs 37C_R

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 3. Clean Up

- 重要な情報を左側に集めるため、一旦 `core` データフレームとして選抜し、その後、join する

In [8]:
rna_seq_all_metadata_core = rna_seq_all_metadata_library_notation.select(
    "sample_heat_id",
    "sample_control_id",
    "sample_name",
    "sample_title",
    # columns with a value in every row
    "experiment_accession",
    "experiment_alias",
    "secondary_sample_accession",
    "study_accession",
    "first_public",
    "study_alias",
    "instrument_platform",
    "instrument_model",
    "library_source",
    "library_layout",
    "sample_molecule_ch1",
    "cell_line",
    "stress treatment temperature",
    "stress treatment time",
    "control treatment temperature",
    "rna_library",
    "normalized_label",
    "item_type",
    "reference_doi",
    "strandness",
    "strandness_evidence"
).rename(
    {
        "cell_line": "sample"
    }
)

display(rna_seq_all_metadata_core)

sample_heat_id,sample_control_id,sample_name,sample_title,experiment_accession,experiment_alias,secondary_sample_accession,study_accession,first_public,study_alias,instrument_platform,instrument_model,library_source,library_layout,sample_molecule_ch1,sample,stress treatment temperature,stress treatment time,control treatment temperature,rna_library,normalized_label,item_type,reference_doi,strandness,strandness_evidence
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR27681984""","""SRR27681986""","""wi-38-htert_40°c_rna-seq_rep1 vs wi-38-htert_37°c_rna-seq_rep1""","""WI-38-hTERT, 40°C, RNA-seq, rep1 vs WI-38-hTERT, 37°C, RNA-seq, rep1""","""SRX23349232 vs SRX23349230""","""GSM8029688_r1 vs GSM8029686_r1""","""SRS20210616 vs SRS20210614""","""PRJNA1067599""","""2024-10-16""","""GSE253874""","""ILLUMINA""","""Illumina HiSeq 2000""","""TRANSCRIPTOMIC""","""SINGLE""","""total RNA""","""WI-38-hTERT""","""40 C""","""120 min.""","""37 C""","""TruSeq Stranded Total RNA Library Prep Kit""","""TruSeq Stranded Total RNA Library Prep Kit""","""library_preparation""","""https://doi.org/10.1111/gtc.13140""","""stranded""","""https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-total-rna-workflow/truseq-stranded-total-rna-workflow-refer…"
"""SRR27681983""","""SRR27681985""","""wi-38-htert_40°c_rna-seq_rep2 vs wi-38-htert_37°c_rna-seq_rep2""","""WI-38-hTERT, 40°C, RNA-seq, rep2 vs WI-38-hTERT, 37°C, RNA-seq, rep2""","""SRX23349233 vs SRX23349231""","""GSM8029689_r1 vs GSM8029687_r1""","""SRS20210615 vs SRS20210617""","""PRJNA1067599""","""2024-10-16""","""GSE253874""","""ILLUMINA""","""Illumina HiSeq 2000""","""TRANSCRIPTOMIC""","""SINGLE""","""total RNA""","""WI-38-hTERT""","""40 C""","""120 min.""","""37 C""","""TruSeq Stranded Total RNA Library Prep Kit""","""TruSeq Stranded Total RNA Library Prep Kit""","""library_preparation""","""https://doi.org/10.1111/gtc.13140""","""stranded""","""https://support.illumina.com/content/dam/illumina-support/documents/documentation/chemistry_documentation/samplepreps_truseq/truseq-stranded-total-rna-workflow/truseq-stranded-total-rna-workflow-refer…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""HRR255913""","""HRR255911""","""RNAseq_Neuron_39_rep1 vs RNAseq_Neuron_37_rep1""","""RNAseq_Neuron_39_rep1 vs RNAseq_Neuron_37_rep1""","""HRX214100 vs HRX214098""","""RNAseq_Neuron_39 vs RNAseq_Neuron_37""","""HRS192117 vs HRS192116""","""PRJCA005446""","""2019-10-30""","""PRJCA005446""","""ILLUMINA""","""Illumina HiSeq X Ten""","""GENOMIC""","""PAIRED""","""total RNA""","""human neuron (hNeuron)""","""39 C""","""2880 min.""","""37 C""","""NEBNext® UltraTM Directional RNA Library Prep Kit""","""NEBNext Ultra Directional RNA Library Prep Kit for Illumina""","""library_preparation""","""https://doi.org/10.1007/s13238-021-00887-y""","""stranded""","""https://www.neb.com/en/products/e7420-nebnext-ultra-directional-rna-library-prep-kit-for-illumina"""
"""HRR255914""","""HRR255912""","""RNAseq_Neuron_39_rep2 vs RNAseq_Neuron_37_rep2""","""RNAseq_Neuron_39_rep2 vs RNAseq_Neuron_37_rep2""","""HRX214101 vs HRX214099""","""RNAseq_Neuron_39 vs RNAseq_Neuron_37""","""HRS192117 vs HRS192116""","""PRJCA005446""","""2019-10-30""","""PRJCA005446""","""ILLUMINA""","""Illumina HiSeq X Ten""","""GENOMIC""","""PAIRED""","""total RNA""","""human neuron (hNeuron)""","""39 C""","""2880 min.""","""37 C""","""NEBNext® UltraTM Directional RNA Library Prep Kit""","""NEBNext Ultra Directional RNA Library Prep Kit for Illumina""","""library_preparation""","""https://doi.org/10.1007/s13238-021-00887-y""","""stranded""","""https://www.neb.com/en/products/e7420-nebnext-ultra-directional-rna-library-prep-kit-for-illumina"""


In [9]:
rna_seq_all_metadata_core.write_csv(
    f"../../Data/metadata/metadata_{VERSION}/rna_seq_all_metadata_simple_{execute_date}.tsv",
    separator="\t"
)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 4. JOIN

- `rna_seq_all_metadata_core` をキーに、`rna_seq_all_metadata_library_notation` と `join` する

In [10]:
rna_seq_all_metadata_all_columns = rna_seq_all_metadata_core.join(
    rna_seq_all_metadata_library_notation.rename(
        {
            "cell_line": "sample"
        }
    ),
    on=rna_seq_all_metadata_core.columns, # use every core column as a join key
    how="inner",
    nulls_equal=True, # treat null == null so rows with a null key still match
    validate="1:1"
)

rna_seq_all_metadata_all_columns.write_csv(
    f"../../Data/metadata/metadata_{VERSION}/rna_seq_all_metadata_all_columns_{execute_date}.tsv",
    separator="\t"
)

display(rna_seq_all_metadata_all_columns)

sample_heat_id,sample_control_id,sample_name,sample_title,experiment_accession,experiment_alias,secondary_sample_accession,study_accession,first_public,study_alias,instrument_platform,instrument_model,library_source,library_layout,sample_molecule_ch1,sample,stress treatment temperature,stress treatment time,control treatment temperature,rna_library,normalized_label,item_type,reference_doi,strandness,strandness_evidence,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,cell_type,genotype,treatment,mapping,expression_estimation,deg_analysis,assembly,supplementary_files_format_and_content,sample_accession,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,country,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,description,cell_type_right,base_count,library_strategy,status,sample_alias,run_alias,library_construction_protocol,collection_date,cell_line_right,library_name,first_created,sample_data_processing,sample_contact_email,tissue,biological_replicates,tissue_type,protocol,organism,read_type,data_source,srr,srx,sample_contact_department,passage,phenotype,replicate,genome_build,center_name,sample_contact_laboratory,sample_supplementary_file_2,sample_supplementary_file_3,disease,stress,sample_contact_phone,tumor,sex,age,sample_treatment_protocol_ch1,sample_group,cell_model,sample_extract_protocol_ch1,time,library_strategy_right,sample_growth_protocol_ch1,sirna,condition,tissue_cells,antibody,batch,column,library_name_right,sample_characteristics_ch1,centrifuge,strain,fraction,enzyme_chemical_treatment,rt_condition,strain_right,treatment_conditions,molecule,isolate,isolate_right,dosage,experiment,timepoint,vector,sirna_sgrna,sra_file_role,fastq_file_role,tag,scientific_name,tax_id,accession,collection_date_end,collection_date_start,submitted_galaxy,submitted_ftp,submitted_aspera,submitted_md5,checklist,submitted_bytes,nominal_sdev,broker_name,datahub,nominal_length,submitted_read_type,submitted_format,bam_ftp,bam_bytes,dev_stage,bam_aspera,bam_galaxy,bam_file_role,submitted_file_role,recovery treatment temperature,recovery treatment time,sample_series_id,biosample,sra,fastq_galaxy,fastq_ftp,fastq_aspera,tax_lineage,Read length for mate1(bp),Read length for mate 2(bp),Individual Accession,Biomaterial Provider
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str
"""SRR27681984""","""SRR27681986""","""wi-38-htert_40°c_rna-seq_rep1 vs wi-38-htert_37°c_rna-seq_rep1""","""WI-38-hTERT, 40°C, RNA-seq, rep1 vs WI-38-hTERT, 37°C, RNA-seq, rep1""","""SRX23349232 vs SRX23349230""","""GSM8029688_r1 vs GSM8029686_r1""","""SRS20210616 vs SRS20210614""","""PRJNA1067599""","""2024-10-16""","""GSE253874""","""ILLUMINA""","""Illumina HiSeq 2000""","""TRANSCRIPTOMIC""","""SING